In [ ]:
import os
import sys
from glob import glob
import time

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# to import logger, VR-wide constants and device paths
sys.path.append(os.path.join('.', ".."))
sys.path.append(os.path.join('..', ".."))
from baseVR.base_logger import CustomLogger as Logger
from baseVR.base_functionality import device_paths

nas_dir = device_paths()[0]


In [ ]:
L = Logger()
L.init_logger(None, None, "DEBUG")

In [ ]:
nas_dir = device_paths()[0]
implant_name = "260611_MEA1K24_S1688pad14shankB5"
headstage_name = "MEA1K24"
subdir = f"devices/headstage_devices/{headstage_name}/recordings"

In [ ]:
def load_impedance_data(rec_dir, nas_dir, subdir, implant_name):
    """Load and merge impedance data with implant mapping."""
    fullfname = os.path.join(nas_dir, "devices", "implant_devices", implant_name, "bonding", 
                            f"bonding_mapping_{implant_name}.csv")
    implant_mapping = pd.read_csv(fullfname)
    
    full_recdir = os.path.join(nas_dir, subdir, rec_dir)
    aggr_df = pd.read_csv(os.path.join(full_recdir, "processed", "all_impedance.csv"))
    aggr_df.rename(columns={"electrode": "mea1k_el"}, inplace=True)
    
    # Merge once with implant mapping
    aggr_df = pd.merge(aggr_df, implant_mapping[['mea1k_el', 'shank_id', 'mea1k_connectivity', 'pad_metal']], on='mea1k_el', how='left')
    aggr_df.rename(columns={'mea1k_el': 'electrode'}, inplace=True)
    
    return aggr_df, implant_mapping, full_recdir


In [ ]:
rec_dir = f"2026-06-28_07.40_Bond4_ext200uA_check"
# rec_dir = f"2026-06-28_18.44_Bond4_ext200uA_singles"
# rec_dir = f"2026-06-28_19.44_Bond4_ext200uA_singles_votlagemode"
# rec_dir = f"2026-06-28_21.10_Bond4_ext200uA_singles_currentemode"

# rec_dir = f"2026-06-29_07.10_Bond4_ext200uA_singles_currentemode"
# rec_dir = f"2026-06-29_09.26_Bond4_ext200uA_singles_currentemode" # now using offset

rec_dir = f"2026-06-29_10.48_Bond4_ext200uA_singles"


_, implant_mapping, full_recdir = load_impedance_data(f"2026-06-24_20.35_bond4_Imp_GND_PT3_1.8mm_LC", nas_dir, subdir, implant_name)
extr_connectivity = pd.read_csv(os.path.join(nas_dir, subdir, rec_dir, "processed", "extr_connectivity.csv")).rename(columns={"el": 'mea1k_el'})

# join on mea1k_el to get shank_id and pad_metal
aggr_df = pd.merge(extr_connectivity, implant_mapping[['mea1k_el', 'shank_id', 'pad_metal', 'mea1k_connectivity']], on='mea1k_el', how='left')
display(aggr_df)

In [ ]:
plt.close('all')
# plt.scatter(aggr_df['mea1k_connectivity'], (aggr_df['ampl_mV']*1000) /0.2, s=5, alpha=0.5)
plt.scatter(aggr_df['mea1k_connectivity'], (aggr_df['ampl_mV']), s=5, alpha=0.5)
# add labels
plt.xlabel("Connectivity (external sine voltage)")
plt.ylabel("Sensing 0.2uA current")

In [ ]:
aggr_df
x, y = aggr_df['mea1k_el'].values %220, aggr_df['mea1k_el'].values // 220

In [ ]:
grid = np.zeros((120, 220))
grid[y, x] = aggr_df['ampl_mV'].values
plt.figure(figsize=(32, 26))
plt.imshow(grid, cmap='gray_r', origin='lower')

In [ ]:
nas_dir = device_paths()[0]
implant_name = "260602_MEA1K23_S844pad8shankB5"
headstage_name = "MEA1K23"
subdir = f"devices/headstage_devices/{headstage_name}/recordings"

In [ ]:
rec_name1 = f"2026-06-30_14.42_8Sh4SilverPaint_Ext0.2uA_PT1_1.7mm_+MANYd_S1-8no7,8/"
rec_name2 = f"2026-06-30_14.44_8Sh4SilverPaint_Ext10mV_PT1_1.7mm_+MANYd_S1-8no7,8/"
# singles
rec_name3 = f"2026-07-01_15.46_8Sh4SilverPaint_Ext0.2uA_Singles_PT1_1.7mm_+MANYd_S1-8no7,8"
#  /mnt/SpatialSequenceLearning/devices/headstage_devices/MEA1K23/recordings/2026-06-30_10.58_8Sh4SilverPaint_ImpLate_7removed_LC/processed/all_impedance.csv 
rec_name4 = f"2026-06-30_10.58_8Sh4SilverPaint_ImpLate_7removed_LC"

par_current = pd.read_csv(os.path.join(nas_dir, subdir, rec_name1, "processed", "extr_connectivity.csv")).rename(columns={"el": 'mea1k_el'}).set_index('mea1k_el')
par_voltage = pd.read_csv(os.path.join(nas_dir, subdir, rec_name2, "processed", "extr_connectivity.csv")).rename(columns={"el": 'mea1k_el'}).set_index('mea1k_el')
singles_current = pd.read_csv(os.path.join(nas_dir, subdir, rec_name3, "processed", "extr_connectivity.csv")).rename(columns={"el": 'mea1k_el'}).set_index('mea1k_el')
impedance = pd.read_csv(os.path.join(nas_dir, subdir, rec_name4, "processed", "all_impedance.csv")).set_index('electrode')

electrodes = singles_current.index

impedance

In [ ]:
# compare
sc = plt.scatter(par_current.loc[electrodes, 'ampl_mV'], singles_current.loc[electrodes, 'ampl_mV'], 
            s=5, alpha=0.5, c=impedance.reindex(electrodes)['impedance_Ohm'], cmap='viridis', vmin=0, vmax=1_000_000)
plt.xlabel('Parallel Current (mV)')
plt.ylabel('Singles Current (mV)')
plt.title('Comparison of Extracellular Recordings')
# identy line
plt.plot([0, 40], [0, 40],  linestyle='--', linewidth=1, color='gray')
# add colorbar
cbar = plt.colorbar(sc)
cbar.set_label('Impedance (kOhm)')
plt.show()

# compare
plt.scatter(par_current.loc[electrodes, 'ampl_mV'], par_voltage.loc[electrodes, 'ampl_mV'], 
            s=5, alpha=0.5, c=impedance.reindex(electrodes)['impedance_Ohm'], cmap='viridis', vmin=0, vmax=1_000_000)
plt.xlabel('Parallel Current (mV)')
plt.ylabel('Parallel Voltage (mV)')
plt.plot([0, 40], [0, 40],  linestyle='--', linewidth=1, color='gray')
plt.title('Comparison of Extracellular Recordings')
plt.show()

plt.scatter(impedance.reindex(electrodes)['connectivity'],  impedance.reindex(electrodes)['impedance_Ohm'], )

